## Tapis and MINT Pipeline
#### 1. Dependencies Installation and Setup Secrets

In [10]:
# Installation of the other standard libraries
%pip install python-dotenv tapipy urllib3

Note: you may need to restart the kernel to use updated packages.


In [24]:
import os, json, requests, urllib3, time
from dotenv import load_dotenv
from tapipy.tapis import Tapis

load_dotenv()
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


DCAT = "https://data-catalog.tacc.mint.isi.edu"
PROVID = "9ef60317-5da5-4050-8bbc-7d6826fee49f" # Change
PROVENANCE_URL = "/provenance/register_provenance"

# Tapis Client
t = Tapis(
    base_url=os.getenv('TAPIS_BASE_URL'), 
    username=os.getenv('TAPIS_USERNAME'), 
    password=os.getenv('TAPIS_PASSWORD'), 
    verify=False
)
t.get_tokens()

print(f"✅ Configurado para TACC MINT. Provenance: {PROVID}")

✅ Configurado para TACC MINT. Provenance: 9ef60317-5da5-4050-8bbc-7d6826fee49f


In [ ]:
def create_provenance_id(provenance_id):
    provenance_definition = {
        "provenance": {
            "provenance_type": "user",
            "record_id": provenance_id,
            "name": "s_sterling22", 
            "metadata": {
                "contact_information": {
                    "email": "seth@example.com" 
                }
            }
        }
    }
    r = requests.post(f"{DCAT}{PROVENANCE_URL}", json=provenance_definition)
    if r.status_code in [200, 201]:
        print(f"✅ Provenance ID {provenance_id} verificado/creado.")
    else:
        print(f"⚠️ Nota: El Provenance ID ya existe o hubo un error: {r.status_code}")
        
create_provenance_id(PROVID)

#### 2. Logic Registration (MCAT)

In [23]:
def register_run_as_dataset(run_uuid, dataset_name):
    # Lógica simplificada del código de tu jefe
    payload = {
        "datasets": [{
            "provenance_id": PROVID,
            "name": dataset_name,
            "description": f"Tapis Workflow Run {run_uuid}",
            "metadata": {
                "pipeline_id": "hello-world-pipeline",
                "execution_engine": "Tapis-V3",
                "image": "ghcr.io/sethsterling22/tapis-workflow-automation/tapis-dummy-app:latest"
            }
        }]
    }
    
    # Registro de Dataset
    r = requests.post(f"{DCAT}/datasets/register_datasets", json=payload)
    r.raise_for_status()
    dsid = r.json()["datasets"][0]["record_id"]
    
    # Registro de Recurso (El CSV final)
    file_url = f"https://portals.tapis.io/v3/files/content/system/outputs/run_{run_uuid}/result.csv"
    res_payload = {
        "resources": [{
            "provenance_id": PROVID,
            "dataset_id": dsid,
            "name": "analysis_results.csv",
            "url": file_url
        }]
    }
    requests.post(f"{DCAT}/datasets/register_resources", json=res_payload)
    
    # Sincronización
    requests.post(f"{DCAT}/datasets/sync_dataset_metadata", json={"dataset_id": dsid})
    
    print(f"✅ Flujo completo en TACC. Dataset ID: {dsid}")
    return dsid

# Simular registro tras un run
# register_run_as_dataset(run_uuid, "Network_Automation_Results")

🔍 Iniciando descubrimiento de endpoint para MCAT...
Probando: https://api.models.mint.isi.edu/v1/mcat/modelconfigurations -> FALLO (404)
Probando: https://api.models.mint.isi.edu/v1/modelconfigurations -> FALLO (404)
Probando: https://api.models.mint.isi.edu/v1/api/mcat/modelconfigurations -> FALLO (404)
Probando: https://api.models.mint.isi.edu/v1/mcat/s_sterling22/modelconfigurations -> FALLO (404)
❌ No se pudo encontrar un endpoint de registro válido. Verifica la URL base.


#### 3. Implementation and Monitoring (Tapis)

In [ ]:
print(f"🚀 Lanzando Pipeline en Tapis...")
run = t.workflows.runPipeline(group_id=os.getenv('TAPIS_GROUP_ID'), pipeline_id="hello-world-pipeline")
run_uuid = run.uuid

while True:
    res = t.workflows.getPipelineRun(group_id=os.getenv('TAPIS_GROUP_ID'), pipeline_id="hello-world-pipeline", pipeline_run_uuid=run_uuid)
    print(f"[{time.strftime('%H:%M:%S')}] Status: {res.status}")
    if res.status == 'completed': break
    if res.status in ['failed', 'cancelled']: raise Exception("El pipeline de Tapis falló.")
    time.sleep(10)

print(f"✅ Pipeline completado. UUID: {run_uuid}")

#### 4. Results Log (DCAT)

In [16]:
def register_output_to_dcat(dataset_name, file_url, run_uuid):
    # 1. Registro del Dataset
    ds_payload = {
        "datasets": [{
            "provenance_id": PROVID,
            "name": dataset_name,
            "description": "Dataset procesado automáticamente mediante Tapis Workflow",
            "metadata": {"source": "tapis-workflow-automation", "tapis_run_uuid": run_uuid}
        }]
    }
    
    ds_resp = requests.post(f"{DCAT_URL}/datasets/register_datasets", json=ds_payload)
    dsid = ds_resp.json()["datasets"][0]["record_id"]
    
    # 2. Registro del Recurso Físico
    res_payload = {
        "resources": [{
            "provenance_id": PROVID,
            "dataset_id": dsid,
            "name": f"result_{run_uuid}.csv",
            "url": file_url,
            "metadata": {"storage_provider": "tapis"}
        }]
    }
    requests.post(f"{DCAT_URL}/datasets/register_resources", json=res_payload)
    
    print(f"✅ Registro en DCAT exitoso. Dataset ID: {dsid}")
    return dsid

# Ejecución con datos genéricos
file_url = f"https://portals.tapis.io/v3/files/content/system/outputs/run_{run_uuid}/result.csv"
DATASET_ID = register_output_to_dcat("Network_Analysis_Output", file_url, run_uuid)

✅ Configurado para enviar datos a: https://api.data.mint.isi.edu/v1


#### 5. Final Sync

In [ ]:
def sync_mint(dsid):
    payload = {"dataset_id": dsid}
    r = requests.post(f"{DCAT_URL}/datasets/sync_dataset_metadata", json=payload)
    if r.status_code == 200:
        print("🔄 Metadatos sincronizados. El ETL ha finalizado y el modelo principal puede iniciar.")

sync_mint(DATASET_ID)